# Simulate data 
... and compress it with relative binning.

Assumes you have run the previous notebook: `1-generate_parameters.ipynb`

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
import sys
sys.path.append('../../cogwheel/')
sys.path.append('..')

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal
lal.swig_redirect_standard_output_error(False)

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from cogwheel import data

from cogwheel_machine.waveform_model import PhenomenologicalWaveformGenerator
from cogwheel_machine.simulation import DataPreprocessor, Simulator

In [ ]:
# Assumes you have run the previous notebook: 1-generate_parameters.ipynb
sim_dir = Path('../data/set_0')
simulation_parameters = pd.read_feather(sim_dir/'simulation_parameters.feather')

In [ ]:
event_data_kwargs = {
    'eventname': '',
    'duration': 32,
    'detector_names': 'HL',
    'asd_funcs': ['asd_H_O3', 'asd_L_O3'],
    'tgps': 0.,
    'tcoarse': 0.,
    }
approximant = 'IMRPhenomD'

In [ ]:
simulator = Simulator(event_data_kwargs, approximant)

In [ ]:
dummy_event_data = data.EventData.gaussian_noise(**event_data_kwargs)
waveform_model = PhenomenologicalWaveformGenerator.from_event_data(
    event_data=dummy_event_data,
    pn_phase_tol=0.1)

data_preprocessor = DataPreprocessor(
    waveform_model,
    pn_phase_tol_compression=1.0  # When we have a better compression algorithm (Joshua's autoencoder) we can decrease this
)

In [ ]:
%%time
# Demo: generate compressed data for a single set of parameters
parameters = simulation_parameters.iloc[0]
compressed_data = data_preprocessor.preprocess_data(
    **simulator.generate_data_and_reference_waveform(parameters))

In [ ]:
simulator.generate_data_and_reference_waveform(parameters)

In [ ]:
plt.figure()
plt.plot(compressed_data)